In [0]:
# Load the prepared 1-day-ahead training dataset
training_df = spark.table(
    "oil_gas_demo.gold.demand_training_features_1day"
)

print("Total rows:", training_df.count())

display(training_df.limit(5)) 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


dates_df = (
    training_df
    .select("target_date")
    .distinct()
    .orderBy("target_date")
)

total_dates = dates_df.count()
train_date_count = int(total_dates * 0.80)

date_window = Window.orderBy("target_date")


# transaction date , product name, and distination city pe composite key 
cutoff_date = (
    dates_df
    .withColumn(
        "date_number",
        F.row_number().over(date_window)
    )
    .filter(F.col("date_number") == train_date_count)
    .select("target_date")
    .first()["target_date"]
)


# 3. Split into training and testing datasets
train_df = training_df.filter(
    F.col("target_date") <= F.lit(cutoff_date)
)

test_df = training_df.filter(
    F.col("target_date") > F.lit(cutoff_date)
)

# 4. Display split results
print("Total unique dates:", total_dates)
print("Training cutoff date:", cutoff_date)
print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

In [0]:

feature_cols = [
    "total_demand",
    "avg_unit_price",
    "total_inventory",
    "day_of_week",
    "month",
    "demand_lag_1",
    "demand_lag_7",
    "rolling_avg_7",
    "product_name",
    "destination_city"
]

# Convert Spark DataFrames to pandas
train_pd = train_df.select(
    feature_cols + ["target_demand"]
).toPandas()

test_pd = test_df.select(
    feature_cols + ["target_demand"]
).toPandas()

# Separate input features (X) and target (y)
X_train = train_pd[feature_cols]
y_train = train_pd["target_demand"]

X_test = test_pd[feature_cols]
y_test = test_pd["target_demand"]

# Verify the data shapes
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Numerical columns
numeric_features = [
    "total_demand",
    "avg_unit_price",
    "total_inventory",
    "day_of_week",
    "month",
    "demand_lag_1",
    "demand_lag_7",
    "rolling_avg_7"
]

# Categorical columns
categorical_features = [
    "product_name",
    "destination_city"
]

# Create the preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numeric_features
        )
    ]
)

# Learn categories from training data and transform it
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same transformation to test data
X_test_processed = preprocessor.transform(X_test)

# Check the output
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# 1. Create the Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# 2. Train the model
rf_model.fit(
    X_train_processed,
    y_train
)

print("Model training completed!")

# 3. Predict next-day demand on test data
y_pred = rf_model.predict(X_test_processed)

# 4. Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

# 5. Display results
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))

In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Baseline: predict the average demand from training data
baseline_value = y_train.mean()

baseline_pred = np.full(
    shape=len(y_test),
    fill_value=baseline_value
)

# Evaluate the baseline
baseline_mae = mean_absolute_error(
    y_test, baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_pred)
)

baseline_r2 = r2_score(
    y_test, baseline_pred
)

# Compare baseline with Random Forest
print("Baseline prediction:", round(baseline_value, 2))

print("\n--- Baseline Model ---")
print("MAE:", round(baseline_mae, 2))
print("RMSE:", round(baseline_rmse, 2))
print("R2:", round(baseline_r2, 4))

print("\n--- Random Forest Model ---")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 4))

In [0]:
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Predict tomorrow's demand using today's demand
naive_pred = X_test["demand_lag_1"].to_numpy()

# Actual next-day demand
actual = np.asarray(y_test)

# Evaluate the naive forecast
naive_mae = mean_absolute_error(actual, naive_pred)

naive_rmse = np.sqrt(
    mean_squared_error(actual, naive_pred)
)

naive_r2 = r2_score(actual, naive_pred)

print("--- Naive Forecast: Tomorrow = Today ---")
print("MAE:", round(naive_mae, 2))
print("RMSE:", round(naive_rmse, 2))
print("R2:", round(naive_r2, 4))

In [0]:
import pandas as pd
import numpy as np

# Compare actual demand with Random Forest predictions
comparison_df = pd.DataFrame({
    "actual_demand": np.asarray(y_test),
    "rf_prediction": np.asarray(y_pred)
})

print("--- Actual Demand Statistics ---")
print(comparison_df["actual_demand"].describe())

print("\n--- Random Forest Prediction Statistics ---")
print(comparison_df["rf_prediction"].describe())

print("\n--- Actual vs Predicted Correlation ---")
print(
    round(
        comparison_df["actual_demand"].corr(
            comparison_df["rf_prediction"]
        ),
        4
    )
)

In [0]:
from pyspark.sql import functions as F

# Check demand variation by product and city
group_summary = (
    test_df
    .groupBy("product_name", "destination_city")
    .agg(
        F.count("target_demand").alias("row_count"),
        F.mean("target_demand").alias("mean_demand"),
        F.stddev("target_demand").alias("std_demand"),
        F.min("target_demand").alias("min_demand"),
        F.max("target_demand").alias("max_demand")
    )
    .orderBy(F.desc("mean_demand"))
)

display(group_summary.limit(20))

In [0]:
import mlflow

# 1. Create or select an MLflow experiment
mlflow.set_experiment(
    "/Shared/Demand_Forecasting_1Day"
)

# 2. Start a new MLflow run
with mlflow.start_run(
    run_name="RandomForest_1Day_Baseline"
):

    # 3. Log model parameters
    mlflow.log_params({
        "model_type": "RandomForestRegressor",
        "n_estimators": 200,
        "max_depth": 15,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "random_state": 42,
        "forecast_horizon": "1_day"
    })

    # 4. Log the evaluation metrics
    mlflow.log_metrics({
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2)
    })

    print("Parameters and metrics logged to MLflow.")

In [0]:
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from mlflow.models import infer_signature

# Step 1: Combine preprocessing and the trained model
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

# Step 2: Create the model signature
input_example = X_train.head(5)

predictions_example = full_pipeline.predict(input_example)

signature = infer_signature(
    input_example,
    predictions_example
)

# Step 3: Log the complete pipeline in a new MLflow run
with mlflow.start_run(
    run_name="RandomForest_1Day_Complete_Pipeline"
):

    mlflow.sklearn.log_model(
        sk_model=full_pipeline,
        artifact_path="model",
        signature=signature,
        input_example=input_example
    )

    print("Complete model pipeline logged successfully!")

In [0]:
import mlflow.sklearn

# Save the trained model in MLflow
mlflow.sklearn.log_model(
    sk_model=rf_model,
    artifact_path="model"
)

print("Model saved in MLflow!")

In [0]:
import mlflow
import mlflow.sklearn

# Save the model and get its location
with mlflow.start_run(run_name="Save_and_Load_Model"):

    model_info = mlflow.sklearn.log_model(
        sk_model=rf_model,
        artifact_path="model"
    )

# Load the saved model
loaded_model = mlflow.sklearn.load_model(
    model_info.model_uri
)

print("Model loaded successfully!")

In [0]:

X_test_processed = preprocessor.transform(X_test)


predictions = loaded_model.predict(X_test_processed)

print("Next-day demand predictions:")

print(predictions[:5])

In [0]:
from sklearn.metrics import mean_absolute_error


loaded_mae = mean_absolute_error(y_test, predictions)

print("Loaded Model MAE:", round(loaded_mae, 2))
print("Original Model MAE:", round(mae, 2))

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature


sample_predictions = full_pipeline.predict(X_train)

signature = infer_signature(
    X_train,
    sample_predictions
)


with mlflow.start_run(run_name="RF_Model_With_Signature"):

    mlflow.sklearn.log_model(
        sk_model=full_pipeline,
        artifact_path="model",
        signature=signature
    )

print("Model logged with input and output signature.")

In [0]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get the best run (RF_Model_With_Signature)
experiment_id = "3607065659861312"
runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.mlflow.runName = 'RF_Model_With_Signature'",
    order_by=["start_time DESC"],
    max_results=1
)

if runs:
    best_run = runs[0]
    run_id = best_run.info.run_id
    
    print(f"Registering model from run: {run_id}")
    print(f"Run name: {best_run.info.run_name}")
    
    # Register the model to Unity Catalog
    model_uri = f"runs:/{run_id}/model"
    
    model_version = mlflow.register_model(
        model_uri=model_uri,
        name="oil_gas_demo.gold.demand_forecast_1day_model",
        tags={
            "model_type": "RandomForestRegressor",
            "forecast_horizon": "1_day",
            "mae": "3513.13",
            "status": "baseline",
            "version_notes": "Initial baseline model with 10 features"
        }
    )
    
    print(f"\n Model registered successfully!")
    print(f"Model Name: {model_version.name}")
    print(f"Version: {model_version.version}")
    print(f"Status: {model_version.current_stage}")
    
else:
    print(" Could not find run to register. Please check run name.")